# Phase A0 — Seed prefilter

Runs `cap-prefilter-seeds` to drop multi-face + deterministic-failure seeds before Stage A. Outputs `confirmed_seeds.json` consumed by Stage A.


In [ ]:
%pip install --force-reinstall --no-deps mediapipe==0.10.20
%pip install ftfy regex torchsde
%pip install /Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline

In [ ]:
dbutils.library.restartPython()

In [ ]:
import os
from pathlib import Path

# Widget params (with defaults)
dbutils.widgets.text('config_path', '/Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline/configs/full.yaml')
dbutils.widgets.text('candidates', '250')
dbutils.widgets.text('target_confirmed', '200')
dbutils.widgets.text('threshold', '0.5')
dbutils.widgets.text('output_dir', '/Volumes/ds_work/alenj00/cap_cache/runs/full/seed_filter')

CONFIG = dbutils.widgets.get('config_path')
CANDIDATES = int(dbutils.widgets.get('candidates'))
TARGET = int(dbutils.widgets.get('target_confirmed'))
THRESHOLD = float(dbutils.widgets.get('threshold'))
OUTPUT_DIR = dbutils.widgets.get('output_dir')

# Idempotency: if confirmed_seeds.json already exists, skip.
out_file = Path(OUTPUT_DIR) / 'confirmed_seeds.json'
if out_file.exists():
  import json as _json
  ids = _json.loads(out_file.read_text())
  msg = f'SKIP: {out_file} already exists with {len(ids)} confirmed seeds'
  print(msg)
  dbutils.notebook.exit(msg)
print(f'Will run prefilter: {CANDIDATES} candidates → ≥ {TARGET} confirmed')

In [ ]:
import os
os.environ['HF_HOME'] = '/local_disk0/hf_cache'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/local_disk0/hf_cache'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TORCH_EXTENSIONS_DIR'] = '/dev/shm/torch_extensions'
os.environ['TMPDIR'] = '/dev/shm/tmp'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from pathlib import Path
Path('/dev/shm/torch_extensions').mkdir(parents=True, exist_ok=True)
Path('/dev/shm/tmp').mkdir(parents=True, exist_ok=True)

HF_TOKEN = dbutils.secrets.get(scope='cap-secrets', key='hf-token')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

PULID_SRC = '/Volumes/ds_work/alenj00/cap_cache/pulid_src'
import sys
if PULID_SRC not in sys.path:
  sys.path.insert(0, PULID_SRC)

In [ ]:
from click.testing import CliRunner
from cap.cli.prefilter_seeds import main as prefilter_cli

result = CliRunner().invoke(prefilter_cli, [
  '--config', CONFIG,
  '--candidates', str(CANDIDATES),
  '--target-confirmed', str(TARGET),
  '--threshold', str(THRESHOLD),
  '--output-dir', OUTPUT_DIR,
  '--cache-dir', '/local_disk0/hf_cache',
  '--pulid-src', PULID_SRC,
  '--hf-token', HF_TOKEN,
  '--mode', 'full',
], catch_exceptions=False, standalone_mode=False)
print('CLI output (last 2000 chars):')
print(result.output[-2000:])

In [ ]:
import json as _json
from pathlib import Path
out_file = Path(OUTPUT_DIR) / 'confirmed_seeds.json'
ids = _json.loads(out_file.read_text())
msg = f'Prefilter complete: {len(ids)} confirmed seeds → {out_file}'
print(msg)
dbutils.notebook.exit(msg)